# SeedVR2 to 1920x1080 (stage 3 of the three-stage route)

Run by `scripts/upscale.py`. numz/ComfyUI-SeedVR2_VideoUpscaler (Apache-2.0), pinned commit, through its
own CLI. `--prepend_frames 4` softens the first frames but the CLI does not remove them (2026-10-30
pilot: 158 in, 162 out), so they are trimmed here, then each clip is scaled to a 1080 short side and
centre-cropped to exactly 1920x1080. The frame count out must equal the frame count in.

In [ ]:
import os, subprocess, sys, time, json, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), "this notebook needs a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0), "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "torch", torch.__version__)
REPO = Path("/content/seedvr2_videoupscaler")
COMMIT = "4490bd1f482e026674543386bb2a4d176da245b9"
if not (REPO / ".setup_done").exists():
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", COMMIT], check=True)
    # Colab's own torch stays; only the other requirements are installed.
    reqs = [line.strip() for line in (REPO / "requirements.txt").read_text().splitlines() if line.strip() and not line.startswith(("torch", "#"))]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
    (REPO / ".setup_done").touch()
print("SeedVR2 at", COMMIT)

In [ ]:
def frames(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_frames", "-show_entries", "stream=nb_read_frames",
                          "-of", "csv=p=0", str(path)], capture_output=True, text=True, check=True)
    return int(out.stdout.strip())

SRC = Path(os.environ.get("S3_INPUT_DIR", "/content/s3_in"))
DST = Path(os.environ.get("S3_OUTPUT_DIR", "/content/s3_out"))
MODEL = os.environ.get("S3_DIT_MODEL", "seedvr2_ema_3b_fp16.safetensors")
BATCH = int(os.environ.get("S3_BATCH_SIZE", "81"))
PREPEND = int(os.environ.get("S3_PREPEND_FRAMES", "4"))
assert BATCH % 4 == 1, "SeedVR2 batches are 4n+1 frames"
SRC.mkdir(parents=True, exist_ok=True)
DST.mkdir(parents=True, exist_ok=True)
# The CLI's upload cannot create folders, so clips arrive as /content/s3in__<name>.mp4.
for uploaded in Path("/content").glob("s3in__*.mp4"):
    shutil.move(str(uploaded), SRC / uploaded.name.removeprefix("s3in__"))
clips = sorted(SRC.glob("*.mp4"))
assert clips, f"no clips in {SRC}"
for clip in clips:
    raw = DST / f"{clip.stem}_raw.mp4"
    started = time.monotonic()
    command = [sys.executable, str(REPO / "inference_cli.py"), str(clip), "--output", str(raw), "--output_format", "mp4",
               "--video_backend", "ffmpeg", "--dit_model", MODEL, "--model_dir", "/content/models/SEEDVR2",
               "--resolution", "1080", "--batch_size", str(BATCH), "--temporal_overlap", "3", "--prepend_frames", str(PREPEND),
               "--color_correction", "lab", "--seed", "42", "--vae_encode_tiled", "--vae_decode_tiled"]
    proc = subprocess.run(command, cwd=REPO, capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-3000:]); print(proc.stderr[-3000:])
        raise RuntimeError(f"SeedVR2 failed on {clip.name} (exit {proc.returncode})")
    seconds = time.monotonic() - started
    expected, produced = frames(clip), frames(raw)
    trim = produced - expected
    assert trim in (0, PREPEND), f"{clip.name}: {expected} frames in, {produced} out"
    final = DST / f"{clip.stem}_1080.mp4"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(raw), "-vf",
                    f"trim=start_frame={trim},setpts=PTS-STARTPTS,scale=-2:1080:flags=lanczos,crop=1920:1080",
                    "-c:v", "libx264", "-crf", "16", "-pix_fmt", "yuv420p", "-an", str(final)], check=True)
    assert frames(final) == expected, f"{final.name}: frame count changed"
    raw.unlink()
    print(f"SEEDVR2_SECONDS: {clip.name} {seconds:.1f}")
subprocess.run(["tar", "-cf", "/content/s3_out.tar", "-C", str(DST), *[f"{clip.stem}_1080.mp4" for clip in clips]], check=True)
print("S3_TAR: /content/s3_out.tar")